In [4]:
# Load API keys and configuration values from the .env file
from dotenv import load_dotenv
import os

In [5]:
# Override existing environment values so the latest local config is used
load_dotenv(override=True)

True

In [6]:
# Check whether the Gemini API key is available without printing the secret.
gemini_api_key = os.getenv("GEMINI_API_KEY")

if gemini_api_key is None:
    print("GEMINI_API_KEY is not set in the environment variables.")
else:
    print("GEMINI_API_KEY is set successfully.")

GEMINI_API_KEY is set successfully.


In [7]:
# Import the Gemini client library for agent interactions
from google import genai

In [8]:
# Create the Gemini client using the loaded API key
client = genai.Client(api_key=gemini_api_key)
print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [9]:
# Send a prompt to Gemini and print the agent's response
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="What is capital of India?"
)

print("Response from Gemini API:")
print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Response from Gemini API:
The capital of India is **New Delhi**.


In [89]:
# Generate a strong IQ-style question for the next step in the agent workflow
user_prompt = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=user_prompt
)
print("Response from Gemini API:")
print(response.text)

response_questions = response.text

Response from Gemini API:
Three perfectly logical logicians—A, B, and C—are each assigned a secret positive integer. They are told that one of their numbers is the sum of the other two. Each logician can see the other two numbers, but not their own. They are asked in sequence (A, then B, then C, then A, then B...) if they know their own number.

Round 1: 
A says, "No." 
B says, "No." 
C says, "No."

Round 2: 
A says, "No." 
B says, "Yes, my number is 15."

What are the numbers assigned to A, B, and C?


In [ ]:
# Use the generated question as input and print the agent's answer
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=response_questions
)
answer = response.text
print(answer)

In [ ]:
from IPython.display import display, Markdown
display(Markdown(answer))

To find the two hidden integers $x$ and $y$ (where $1 < x < y$ and $x + y \le 100$), we analyze the conversation step-by-step.

---

### Step 1: S says, *"I know that you cannot know the two numbers."*

For S to be certain that P does not know the numbers, **every** possible pair $(a, b)$ that adds up to S's sum $s = a + b$ must produce a product $p = a \cdot b$ that can be factored into at least two different valid pairs.

1. **Even sums are ruled out:** By Goldbach's Conjecture (which holds for numbers $\le 100$), every even number can be expressed as the sum of two primes $p_1 + p_2$. If $x$ and $y$ were primes, their product $p_1 \cdot p_2$ would have only one valid factorization, allowing P to know the numbers immediately. Thus, $s$ cannot be even.
2. **Sums of the form $2 + \text{prime}$ are ruled out:** If $s = 2 + \text{prime}$, the product $2 \cdot \text{prime}$ would have a unique factorization into two integers $> 1$, allowing P to know the numbers.
3. **Sums greater than 53 are ruled out:** Any sum $s > 53$ can be written as $53 + k$. Because $53$ is a prime larger than $100/2$, $53 \cdot k$ cannot be factored into any other pair of numbers whose sum is $\le 100$.

Filtering by these criteria leaves only a small set of **candidate sums**:
$$\{11, 17, 23, 27, 29, 35, 37, 41, 47, 51, 53\}$$

---

### Step 2: P replies, *"Now I know the two numbers."*

P now knows that the sum $s$ must be one of the candidate sums listed above. For P to deduce the exact pair, P's product $p$ must have **only one** pair of factors $(a, b)$ whose sum $a + b$ is in the candidate set.

Let's test the possible pairs for $s = 17$:

* $2 + 15 = 17 \implies p = 30 = 5 \times 6 \implies \text{sum } 11 \text{ (in set)}$ $\rightarrow$ Ambiguous for P
* $3 + 14 = 17 \implies p = 42 = 2 \times 21 \implies \text{sum } 23 \text{ (in set)}$ $\rightarrow$ Ambiguous for P
* **$4 + 13 = 17 \implies p = 52$**
  * Factorizations of $52$:
    * $2 \times 26 \implies \text{sum } 28 \text{ (not in candidate set)}$
    * $4 \times 13 \implies \text{sum } 17 \text{ (in candidate set)}$
  * Since $17$ is the *only* sum in the candidate set, **P can uniquely determine the pair is $(4, 13)$**.
* $5 + 12 = 17 \implies p = 60 = 3 \times 20 \implies \text{sum } 23 \text{ (in set)}$ $\rightarrow$ Ambiguous for P
* $6 + 11 = 17 \implies p = 66 = 2 \times 33 \implies \text{sum } 35 \text{ (in set)}$ $\rightarrow$ Ambiguous for P
* $7 + 10 = 17 \implies p = 70 = 2 \times 35 \implies \text{sum } 37 \text{ (in set)}$ $\rightarrow$ Ambiguous for P
* $8 + 9 = 17 \implies p = 72 = 3 \times 24 \implies \text{sum } 27 \text{ (in set)}$ $\rightarrow$ Ambiguous for P

---

### Step 3: S responds, *"Now I know them too."*

S examines all possible pairs that sum to $17$. As shown in Step 2, **$(4, 13)$ is the only pair** whose product leads to a unique candidate sum, allowing P to deduce the answer.

Since S can eliminate all other pairs, S now uniquely identifies the pair as well.

---

### Final Answer:
The two numbers are **$x = 4$** and **$y = 13$**.

In [ ]:
evaluation_prompt= f" Here is the question: {response_questions} and here is the answer: {answer}. Please evaluate the answer and provide a score out of 10, along with a brief explanation for the score."

In [ ]:

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=evaluation_prompt
)

display(Markdown(response.text))

Here is an evaluation of the provided answer:

### **Score: 10 / 10**

---

### **Evaluation and Explanation:**

1. **Correctness (5/5):** 
   * The final answer ($x = 4, y = 13$) is completely correct.
   * The logical deduction at each step is mathematically sound:
     * **Step 1:** Correctly identifies why even sums, sums of $2 + \text{prime}$, and sums $> 53$ are eliminated, arriving at the exact correct list of candidate sums: $\{11, 17, 23, 27, 29, 35, 37, 41, 47, 51, 53\}$.
     * **Step 2:** Accurately tests all partitions of $s = 17$ and demonstrates why $p = 52$ ($4 \times 13$) allows P to eliminate other factor combinations, while all other pairs for $17$ remain ambiguous to P.
     * **Step 3:** Correctly shows how S uses this uniqueness to conclude $(4, 13)$ is the only valid pair for $s = 17$.

2. **Clarity & Structure (5/5):**
   * The response is exceptionally well-organized, breaking down each line of the conversation systematically.
   * Using explicit examples for all partitions of $s = 17$ makes the explanation easy to follow and visually clear.